# HydroShare S3 Data Access

**Purpose**: This notebook demonstrates best practices for accessing and working with HydroShare resource data directly from its S3-backed storage.

In [ ]:
import hsclient
import getpass

## 1. Connecting to HydroShare Files via S3

This section outlines techniques for working with HydroShare data using Python libraries.

In [ ]:
# Sign in to HydroShare using the hsclient library
# This will prompt you to enter your username and password

username = input("Username: ").strip()
password = getpass.getpass("Password for {}: ".format(username))
hs = hsclient.HydroShare(username, password)

Load a HydroShare resource that we're intersted in and have access to.

In [ ]:
# Accessing a specific HydroShare resource by its ID - use an ID of a resource you have access to
resource_id = '5b379c908019453e86f304e0a07c4984'
resource = hs.resource(resource_id)

Now that the resource has been loaded, we can use `hsclient` functions to browse the files. The `hsclient` library can be use to create, modify, and interact with HydroShare resources. It was designed to allow you to write code to do pretty much everything you can do through HydroShare's web user interface. documentation on of all its capabilities can be found at: https://github.com/hydroshare/hsclient.

Let's explore some of the most basic capabilites, such as listing files and exploring their metadata.

In [ ]:
# Print some basic metadata
print(f'Title: {resource.metadata.title}')
print(f'Description: {resource.metadata.abstract}')

In [ ]:
# List the contents of the HydroShare resource
resource.files()

This library can also be used to upload and download files. Section 2 below demonstrates this directly using `hsclient`'s built-in S3 support.

## 2. Interacting with HydroShare Files via S3

This section demonstrates how to interact with HydroShare content using S3. This is particularly useful when working with large data in HydroShare. Directly connecting to data via the S3 API standard enables us to efficiently upload or download data, as well as perform buffered reading (and lazy data loading) using scientific libraries.

For each operation below we show two equivalent ways of doing it:
1. The convenience methods on `Resource` (e.g. `resource.s3_list_objects()`), which `hsclient` exposes directly.
2. Working directly with the underlying `s3fs.S3FileSystem` client and the resource's bucket path, both obtained from the `HydroShare` instance (`hs`).

`hsclient` authenticates with HydroShare's S3 interface for you when you sign in. The `HydroShare` instance (`hs`) exposes this directly through its `s3_client` property, which gives us an authenticated `s3fs.S3FileSystem` instance. No manual key/token retrieval or `s3fs` setup is needed.

To operate on a specific resource's files, we also need that resource's S3 bucket/prefix path, which we can retrieve with `hs.get_resource_path(resource_id)`. Together, `hs.s3_client` and the resource path returned by `get_resource_path()` are all that's needed to perform any S3 file operation (listing, downloading, uploading, deleting, etc.) against a resource's contents — without going through a `Resource` object at all.

In [ ]:
s3_client = hs.s3_client
assert s3_client is not None

resource_s3_path = hs.get_resource_path(resource_id)
print(resource_s3_path)

We now have everything we need for interacting with our data. Below are a few common operations that are useful when working with large datasets and large collections of data.

### Listing/browsing the contents of a HydroShare resource

In [ ]:
# Option 1: using the resource's convenience method
files = resource.s3_list_objects(recursive=True)
for f in files:
    print(f)

In [ ]:
# Option 2: using s3_client and resource_s3_path directly
files = s3_client.find(resource_s3_path)
for f in files:
    print(f)

### Checking the total size of a resource's files

In [ ]:
# Option 1: using the resource's convenience method
print(f'Total size of files is approximately: {resource.s3_client.du(resource.s3_path) / 1e9} GB')

In [ ]:
# Option 2: using s3_client and resource_s3_path directly
print(f'Total size of files is approximately: {s3_client.du(resource_s3_path) / 1e9} GB')

### Downloading a file

In [ ]:
# Option 1: using the resource's convenience method - use a filename the resource contains
resource.s3_file_download('vpu01_medium_range_troute.parquet', '.')

In [ ]:
# Option 2: using s3_client and resource_s3_path directly - use a filename the resource contains
resource_s3_path = resource_s3_path.strip('/')
remote_path = f"{resource_s3_path}/vpu01_medium_range_troute.parquet"
s3_client.get(remote_path, '.')

### Uploading a file

The code below will only work when you have "edit" permissions on the HydroShare resource. To try this, you'll want to create a new HydroShare resource and replace `test.nc` with a file that you have.

In [ ]:
# Option 1: using the resource's convenience method
resource.s3_file_upload('test.nc')

In [ ]:
# Option 2: using s3_client and resource_s3_path directly
local_file = 'test.nc'
remote_path = f"{resource_s3_path}/{local_file}"
s3_client.put(local_file, remote_path)

### Deleting a file

As with uploading, this requires "edit" permissions on the HydroShare resource.

In [ ]:
# Option 1: using the resource's convenience method
resource.s3_file_delete('test.nc')

In [ ]:
# Option 2: using s3_client and resource_s3_path directly
remote_path = f"{resource_s3_path}/test.nc"
s3_client.rm(remote_path)

## 3. Interacting with HydroShare Files via Python AWS S3 CLI

This section demonstrates how to interact with HydroShare content using the AWS S3 CLI utility. This is particularly useful when working with large data stored in HydroShare when Python is not available. One example of when this is useful is when moving data around on cloud servers or compute instances. 

The first thing you'll need to do is install the Amazon S3 Command Line Interfance on your computer. Since installation varies by computing platform, please refer to the official documentation for installation instructions: https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html.

Once installed, you should see something that resembles the following when running the `aws s3` ccommand in your terminal:

```bash
bash-5.3$ aws s3 help 
S3()                                                                      S3()

NAME
       s3 -

DESCRIPTION
       This section explains prominent concepts and notations in the set of
       high-level S3 commands provided.

       If you are looking for the low level S3 commands for the CLI, please
       see the s3api command reference page.

   Path Argument Type
       Whenever using a command, at least one path argument must be specified.
       There are two types of path arguments: LocalPath and S3Uri.

       LocalPath: represents the path of a local file or directory.  It can be
       written as an absolute path or relative path.

       S3Uri: represents the location of a S3 object, prefix, or bucket.  This
       must be written in the form s3://amzn-s3-demo-bucket/mykey where
       amzn-s3-demo-bucket is the specified S3 bucket, mykey is the specified
       S3 key.  The path argument must begin with s3:// in order to denote
       that the path argument refers to a S3 object. Note that prefixes are
       separated by forward slashes. For example, if the S3 object myobject
       had the prefix myprefix, the S3 key would be myprefix/myobject, and if
       the object was in the bucket amzn-s3-demo-bucket, the S3Uri would be
       s3://amzn-s3-demo-bucket/myprefix/myobject.
       
       ...
       
```

Next, we'll want to create a new "profile" that defines a connection to HydroShare. This can be done by running the following. Note that the name "hydroshare" used in the command below is just a label that we're giving our profile. This will be referenced in future commands, so if you choose to use a alternate name make sure you update subsequent commands to reflect that.

```bash
aws configure --profile hydroshare
```

You'll be prompted for your access key and secret, simply enter the values that were collected in the steps above.

Next, we'll configure the endpoint_url by running the following command:

```bash
aws configure set profile.hydroshare.endpoint_url https://s3.hydroshare.org
```

Now we should be able to query data stored in HydroShare using our credentials!

In [ ]:
!aws s3 ls s3://{resource_s3_path} --profile hydroshare

Now you can use any of the `aws` CLI commands. This means that if you're uploading lots of data you can leverage things like `sync` to move data in an efficient manner ;) 